# Data Cleaning & Feature Engineering

This notebook handles:
- Loading and inspecting the HR Employee Attrition dataset
- Missing data handling and duplicate removal
- Synthetic comment generation for sentiment analysis
- Text preprocessing (cleaning, tokenization)
- VADER sentiment analysis
- Categorical standardization (one-hot encoding)
- Feature engineering (ratios, aggregations, risk scoring)
- Export cleaned dataset with engineered features

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For VADER sentiment analysis
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon', quiet=True)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Load the data
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

print("="*80)
print("INITIAL DATA INSPECTION")
print("="*80)
print(f"\nDataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nColumn names and types:")
print(df.dtypes)

print(f"\n\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✓ No missing values")

print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nFirst few rows:")
print(df.head())

INITIAL DATA INSPECTION

Dataset shape: (1470, 35)
Memory usage: 1.02 MB

Column names and types:
Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction   

## Step 1: Load & Inspect Data

## Step 2: Generate Synthetic Comments for Sentiment Analysis

In [3]:
# Generate synthetic employee feedback comments based on satisfaction and attrition
np.random.seed(42)

# Define comment templates based on satisfaction levels and attrition
positive_comments = [
    "Great team environment, very satisfied with my role and growth opportunities.",
    "Excellent work-life balance and supportive management. Loving this job!",
    "Very happy with compensation and career development here.",
    "Outstanding company culture and amazing colleagues. Would recommend!",
    "Fantastic benefits and flexible work arrangements. Really appreciate this.",
    "Love the collaborative environment and innovative projects.",
    "Great company to work for. Management is supportive and fair.",
    "Excellent opportunities for learning and development.",
    "Very satisfied with my position and the team dynamics.",
    "Amazing experience working here. Highly motivated!"
]

neutral_comments = [
    "It's a decent job with normal work environment.",
    "The role is satisfactory but could be better.",
    "Work is fine, management is okay.",
    "Standard work situation, nothing exceptional.",
    "Average work conditions and typical responsibilities.",
    "Job is acceptable, could have more benefits.",
    "Work environment is neutral, neither good nor bad.",
    "Decent workplace but room for improvement.",
    "Standard corporate job with usual pros and cons.",
    "The job meets expectations for the most part."
]

negative_comments = [
    "Frustrated with lack of growth opportunities and poor management.",
    "Work-life balance is terrible, considering leaving soon.",
    "Very dissatisfied with compensation and benefits.",
    "Unhappy with current role and team dynamics.",
    "Poor company culture and insufficient support.",
    "Stressed due to excessive workload and no flexibility.",
    "Management is unsupportive and career progression is stagnant.",
    "Very frustrated with the work environment.",
    "Thinking about moving on due to poor conditions.",
    "Deeply unhappy with this position and company."
]

# Generate comments based on overall satisfaction
df['EmployeeComments'] = df.apply(
    lambda row: np.random.choice(positive_comments) 
        if row[['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']].mean() >= 3.5
        else (np.random.choice(negative_comments) 
            if row[['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']].mean() <= 2
            else np.random.choice(neutral_comments)),
    axis=1
)

print("="*80)
print("SYNTHETIC COMMENTS GENERATED")
print("="*80)
print(f"\nSample comments:\n")
print(df[['EmployeeNumber', 'EmployeeComments', 'JobSatisfaction']].head(10).to_string())
print(f"\nTotal comments generated: {len(df)}")

SYNTHETIC COMMENTS GENERATED

Sample comments:

   EmployeeNumber                                                EmployeeComments  JobSatisfaction
0               1  Management is unsupportive and career progression is stagnant.                4
1               2                   Standard work situation, nothing exceptional.                2
2               4                      Decent workplace but room for improvement.                3
3               5           Average work conditions and typical responsibilities.                3
4               7              Work environment is neutral, neither good nor bad.                2
5               8                   The job meets expectations for the most part.                4
6              10               Very dissatisfied with compensation and benefits.                1
7              11              Work environment is neutral, neither good nor bad.                3
8              12                      Decent workplace but r

## Step 3: Missing Data Handling & Duplicate Removal

In [4]:
print("="*80)
print("DATA QUALITY MANAGEMENT")
print("="*80)

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {len(numerical_cols)} features")

# Check for missing values before handling
missing_before = df.isnull().sum().sum()
print(f"\nMissing values before: {missing_before}")

# Handle missing data
# For numerical columns: use median by department
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        if 'Department' in df.columns:
            df[col] = df.groupby('Department')[col].transform(
                lambda x: x.fillna(x.median())
            )
        else:
            df[col].fillna(df[col].median(), inplace=True)

# For categorical columns: use mode or 'Unknown'
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()
        df[col].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown', inplace=True)

missing_after = df.isnull().sum().sum()
print(f"Missing values after: {missing_after}")

# Remove exact duplicates
print(f"\nDataset shape before duplicate removal: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Dataset shape after duplicate removal: {df.shape}")

print(f"\n✓ Data quality management complete!")

DATA QUALITY MANAGEMENT

Categorical columns: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime', 'EmployeeComments']
Numerical columns: 26 features

Missing values before: 0
Missing values after: 0

Dataset shape before duplicate removal: (1470, 36)
Duplicate rows: 0
Dataset shape after duplicate removal: (1470, 36)

✓ Data quality management complete!


## Step 4: Text Preprocessing

In [5]:
import re

print("="*80)
print("TEXT PREPROCESSING")
print("="*80)

def preprocess_text(text):
    """Clean and preprocess text for sentiment analysis"""
    # Convert to lowercase
    text = text.lower()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove special characters but keep basic punctuation for VADER
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', '', text)
    return text

# Apply preprocessing
df['EmployeeComments_Processed'] = df['EmployeeComments'].apply(preprocess_text)

print(f"\nSample preprocessing results:")
for idx in [0, 1, 2]:
    print(f"\nOriginal: {df['EmployeeComments'].iloc[idx]}")
    print(f"Processed: {df['EmployeeComments_Processed'].iloc[idx]}")

print(f"\n✓ Text preprocessing complete!")

TEXT PREPROCESSING

Sample preprocessing results:

Original: Management is unsupportive and career progression is stagnant.
Processed: management is unsupportive and career progression is stagnant.

Original: Standard work situation, nothing exceptional.
Processed: standard work situation, nothing exceptional.

Original: Decent workplace but room for improvement.
Processed: decent workplace but room for improvement.

✓ Text preprocessing complete!


## Step 5: VADER Sentiment Analysis

In [6]:
print("="*80)
print("VADER SENTIMENT ANALYSIS")
print("="*80)

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Apply VADER to processed comments
df['sentiment_scores'] = df['EmployeeComments_Processed'].apply(
    lambda x: sia.polarity_scores(x)
)

# Extract individual sentiment scores
df['sentiment_compound'] = df['sentiment_scores'].apply(lambda x: x['compound'])
df['sentiment_positive'] = df['sentiment_scores'].apply(lambda x: x['pos'])
df['sentiment_negative'] = df['sentiment_scores'].apply(lambda x: x['neg'])
df['sentiment_neutral'] = df['sentiment_scores'].apply(lambda x: x['neu'])

# Classify sentiment labels
def classify_sentiment(compound_score):
    if compound_score >= 0.05:
        return 'Positive'
    elif compound_score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_label'] = df['sentiment_compound'].apply(classify_sentiment)

print(f"\nSentiment Analysis Results:")
print(f"\nSentiment Distribution:")
print(df['sentiment_label'].value_counts())

print(f"\nSentiment Compound Score Stats:")
print(df['sentiment_compound'].describe())

print(f"\nSample sentiment analysis:")
display_cols = ['EmployeeNumber', 'EmployeeComments', 'sentiment_compound', 'sentiment_label']
print(df[display_cols].head(10).to_string())

# Drop the scores dictionary column
df = df.drop('sentiment_scores', axis=1)

print(f"\n✓ Sentiment analysis complete!")

VADER SENTIMENT ANALYSIS

Sentiment Analysis Results:

Sentiment Distribution:
sentiment_label
Neutral     608
Positive    606
Negative    256
Name: count, dtype: int64

Sentiment Compound Score Stats:
count    1470.000000
mean        0.181678
std         0.426571
min        -0.582400
25%         0.000000
50%         0.000000
75%         0.612400
max         0.908000
Name: sentiment_compound, dtype: float64

Sample sentiment analysis:
   EmployeeNumber                                                EmployeeComments  sentiment_compound sentiment_label
0               1  Management is unsupportive and career progression is stagnant.              0.0000         Neutral
1               2                   Standard work situation, nothing exceptional.              0.0000         Neutral
2               4                      Decent workplace but room for improvement.              0.6124        Positive
3               5           Average work conditions and typical responsibilities.        

## Step 6: Categorical Standardization (One-Hot Encoding)

In [15]:
print("="*80)
print("CATEGORICAL STANDARDIZATION")
print("="*80)

# Identify categorical columns (excluding comments, sentiment results, and numbers)
exclude_cols = ['EmployeeComments', 'EmployeeComments_Processed', 'EmployeeNumber', 
                'sentiment_label', 'sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral']
categorical_cols = [col for col in df.select_dtypes(include=['object']).columns 
                    if col not in exclude_cols]

print(f"\nCategorical columns to encode: {categorical_cols}")
print(f"\nUnique values per column:")
for col in categorical_cols:
    print(f"  {col}: {df[col].nunique()} values - {df[col].unique().tolist()}")

# Store mapping for reference
encoding_mapping = {}
for col in categorical_cols:
    encoding_mapping[col] = df[col].unique().tolist()

# Check for sentiment columns before encoding
print(f"\n\nCOLUMNS BEFORE ENCODING:")
sentiment_cols_before = [col for col in df.columns if 'sentiment' in col.lower()]
print(f"Sentiment columns: {sentiment_cols_before}")
print(f"Total columns: {df.shape[1]}")

# One-hot encode categorical variables
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=False,
    prefix=categorical_cols
)

print(f"\nCOLUMNS AFTER ENCODING:")
sentiment_cols_after = [col for col in df_encoded.columns if 'sentiment' in col.lower()]
print(f"Sentiment columns: {sentiment_cols_after}")
print(f"Total columns: {df_encoded.shape[1]}")

print(f"\nOriginal shape: {df.shape}")
print(f"Shape after one-hot encoding: {df_encoded.shape}")
print(f"New columns created: {df_encoded.shape[1] - df.shape[1]}")

df = df_encoded

print(f"\n✓ One-hot encoding complete!")

CATEGORICAL STANDARDIZATION

Categorical columns to encode: []

Unique values per column:


COLUMNS BEFORE ENCODING:
Sentiment columns: ['sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_label_Negative', 'sentiment_label_Neutral', 'sentiment_label_Positive']
Total columns: 78

COLUMNS AFTER ENCODING:
Sentiment columns: ['sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_label_Negative', 'sentiment_label_Neutral', 'sentiment_label_Positive']
Total columns: 78

Original shape: (1470, 78)
Shape after one-hot encoding: (1470, 78)
New columns created: 0

✓ One-hot encoding complete!


## Step 7: Feature Engineering

In [8]:
print("="*80)
print("FEATURE ENGINEERING")
print("="*80)

# 1. Salary to Income Ratio
df['salary_to_income_ratio'] = df['MonthlyRate'] / (df['MonthlyIncome'] + 1)

# 2. Workload Score
overtime_cols = [col for col in df.columns if 'OverTime' in col]
if overtime_cols:
    over_time_yes = df[[col for col in overtime_cols if 'Yes' in col][0]] if any('Yes' in col for col in overtime_cols) else 0
    df['workload_score'] = over_time_yes + (df['DailyRate'] > df['DailyRate'].median()).astype(int)
else:
    df['workload_score'] = (df['DailyRate'] > df['DailyRate'].median()).astype(int)

# 3. Career Progress Ratio
df['career_progress_ratio'] = (df['YearsAtCompany'] + 1) / (df['TotalWorkingYears'] + 1)

# 4. Promotion Frequency
df['promotion_frequency'] = (df['YearsAtCompany'] + 1) / (df['YearsSinceLastPromotion'] + 1)

# 5. Job Change Frequency
df['job_change_frequency'] = (df['NumCompaniesWorked'] + 1) / (df['TotalWorkingYears'] + 1)

# 6. Satisfaction Index
satisfaction_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 
                     'RelationshipSatisfaction', 'WorkLifeBalance']
df['satisfaction_index'] = df[satisfaction_cols].mean(axis=1)

# 7. Attrition Risk Score
df['attrition_risk_score'] = (
    (4 - df['satisfaction_index']) * 0.5 +
    df['workload_score'] * 0.3 +
    (1 - df['career_progress_ratio']) * 0.2
)

# 8. Attrition Risk Category
def categorize_risk(score):
    if score <= 1:
        return 'Low_Risk'
    elif score <= 2:
        return 'Medium_Risk'
    else:
        return 'High_Risk'

df['attrition_risk_category'] = df['attrition_risk_score'].apply(categorize_risk)

# 9. Average Rating
df['average_rating'] = (df['JobInvolvement'] + df['PerformanceRating']) / 2

# 10. Engagement Potential
df['engagement_potential'] = 5 - df['attrition_risk_score'].clip(0, 5)

print(f"\nEngineered 10 New Features:")
engineered_features = ['salary_to_income_ratio', 'workload_score', 'career_progress_ratio',
                       'promotion_frequency', 'job_change_frequency', 'satisfaction_index',
                       'attrition_risk_score', 'attrition_risk_category', 'average_rating', 'engagement_potential']
for i, feat in enumerate(engineered_features, 1):
    print(f"  {i}. {feat}")

print(f"\nEngineered Features Statistics:")
numeric_engineered = [f for f in engineered_features if f != 'attrition_risk_category']
print(df[numeric_engineered].describe())

print(f"\nAttrition Risk Distribution:")
print(df['attrition_risk_category'].value_counts())

print(f"\n✓ Feature engineering complete!")

FEATURE ENGINEERING



Engineered 10 New Features:
  1. salary_to_income_ratio
  2. workload_score
  3. career_progress_ratio
  4. promotion_frequency
  5. job_change_frequency
  6. satisfaction_index
  7. attrition_risk_score
  8. attrition_risk_category
  9. average_rating
  10. engagement_potential

Engineered Features Statistics:
       salary_to_income_ratio  workload_score  career_progress_ratio  \
count             1470.000000     1470.000000            1470.000000   
mean                 3.342968        0.782313               0.715539   
std                  2.864592        0.672488               0.296687   
min                  0.108958        0.000000               0.043478   
25%                  1.274769        0.000000               0.500000   
50%                  2.519505        1.000000               0.823529   
75%                  4.530805        1.000000               1.000000   
max                 26.731683        2.000000               1.000000   

       promotion_frequency  job_chang

## Step 8: Data Validation & Export

In [13]:
import json

print("="*80)
print("DATA VALIDATION & EXPORT")
print("="*80)

# Validation checks
print(f"\nValidation Checks:")
print(f"  ✓ Null values: {df.isnull().sum().sum()}")
print(f"  ✓ Duplicate rows: {df.duplicated().sum()}")
print(f"  ✓ Dataset shape: {df.shape}")

# Export cleaned data
output_path = '../data/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f"\n✓ Cleaned data exported to: {output_path}")

# Export feature mapping reference
feature_mapping = {
    'encoding_mapping': encoding_mapping,
    'engineered_features': engineered_features,
    'sentiment_features': ['sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_label']
}

mapping_path = '../data/feature_mapping.json'
with open(mapping_path, 'w') as f:
    json.dump(feature_mapping, f, indent=2, default=str)
print(f"✓ Feature mapping reference exported to: {mapping_path}")

print(f"\n✓ All data cleaning steps complete!")

DATA VALIDATION & EXPORT

Validation Checks:
  ✓ Null values: 0
  ✓ Duplicate rows: 0
  ✓ Dataset shape: (1470, 78)

✓ Cleaned data exported to: ../data/cleaned_data.csv
✓ Feature mapping reference exported to: ../data/feature_mapping.json

✓ All data cleaning steps complete!


## Final Summary Report

In [16]:
print("\n" + "="*80)
print("FINAL CLEANING REPORT - PIPELINE COMPLETE")
print("="*80)

# Reconstruct sentiment labels from compound scores for reporting
df['sentiment_label_reconstructed'] = df['sentiment_compound'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

positive_count = (df['sentiment_label_reconstructed'] == 'Positive').sum()
negative_count = (df['sentiment_label_reconstructed'] == 'Negative').sum()
neutral_count = (df['sentiment_label_reconstructed'] == 'Neutral').sum()

print(f"""
✓ Step 1: Data Loaded Successfully
  - Records: {df.shape[0]}
  - Features: {df.shape[1]}

✓ Step 2: Synthetic Comments Generated
  - Comments created for sentiment analysis
  - Based on satisfaction score patterns

✓ Step 3: Missing Data Handled & Duplicates Removed
  - Remaining nulls: {df.isnull().sum().sum()}
  - Duplicate rows: {df.duplicated().sum()}

✓ Step 4: Text Preprocessing Complete
  - Lowercase conversion applied
  - Whitespace and special characters normalized

✓ Step 5: VADER Sentiment Analysis Applied
  - Positive sentiments: {positive_count} ({positive_count/len(df)*100:.1f}%)
  - Negative sentiments: {negative_count} ({negative_count/len(df)*100:.1f}%)
  - Neutral sentiments: {neutral_count} ({neutral_count/len(df)*100:.1f}%)
  - Average sentiment score: {df['sentiment_compound'].mean():.4f}

✓ Step 6: Categorical Variables One-Hot Encoded
  - Categorical columns encoded: {len(categorical_cols)}
  - New binary features created (attrition_risk_category one-hot encoded)

✓ Step 7: 10 Features Engineered
  - Ratio/Score features: 6
  - Category features: 1
  - Composite features: 3

✓ Step 8: Data Validated & Exported
  - Output file: cleaned_data.csv
  - Feature mapping: feature_mapping.json
  - Total columns exported: {df.shape[1]}

""")
print("="*80)
print("READY FOR EDA!")
print("="*80)


FINAL CLEANING REPORT - PIPELINE COMPLETE

✓ Step 1: Data Loaded Successfully
  - Records: 1470
  - Features: 79

✓ Step 2: Synthetic Comments Generated
  - Comments created for sentiment analysis
  - Based on satisfaction score patterns

✓ Step 3: Missing Data Handled & Duplicates Removed
  - Remaining nulls: 0
  - Duplicate rows: 0

✓ Step 4: Text Preprocessing Complete
  - Lowercase conversion applied
  - Whitespace and special characters normalized

✓ Step 5: VADER Sentiment Analysis Applied
  - Positive sentiments: 606 (41.2%)
  - Negative sentiments: 256 (17.4%)
  - Neutral sentiments: 608 (41.4%)
  - Average sentiment score: 0.1817

✓ Step 6: Categorical Variables One-Hot Encoded
  - Categorical columns encoded: 0
  - New binary features created (attrition_risk_category one-hot encoded)

✓ Step 7: 10 Features Engineered
  - Ratio/Score features: 6
  - Category features: 1
  - Composite features: 3

✓ Step 8: Data Validated & Exported
  - Output file: cleaned_data.csv
  - Featur